# Policy decision ensemble — Databricks round 1

Implements prompts G1–G6 of `databricks/ENSEMBLE_PLAN.md`: generate + label ~200k decision states with
Spark (replay labelling, k=8), run data-quality checks, train the blended decision ensemble
(reference `train/policy_train.py`: ridge + 5 bagged numpy MLPs 30→64→64→4 on masked advantages,
split by seed), export numpy-only weights and check parity in a fresh process.

Round-1 results (data `workspace.default.policy_rows` v8, 205,154 rows from seeds d1..d2760, k=8):

| stage | outcome |
|---|---|
| G0 setup | selfplay vs add3 (100 deals) +0.08 bb/deal; `policy_data.selftest()` 25 rows, 0.8 s; serverless Spark 4.2.0 / py3.12 |
| G1+G2 data | 205,154 rows / 2,760 units in ~35 min wall (~6.9 core-hours); ~74 rows/unit, 2.2 s/unit at 4-way |
| G3 quality | 0 broken rows; advantage SD 20–25 chips; noise floor 0.7–3.7 chips; FOLD best on 45–58% of facing-raise spots |
| G4 train | ensemble adv MAE 12.80, best-action 39.2%, offline value w=0: +3.29 → w=1: +4.13 chips/decision (MLflow `poker_policy_ensemble`) |
| G5 export | `bot/policy_weights.npz` (ridge + mlp0..4); fresh-process parity max|diff| = 0 over 1,000 rows; predict 0.038 ms p50/p95 |
| G6 handover | branch `db-policy-model` |

In [0]:
# G0: setup check — repo on sys.path, warmup, selfplay vs add3, policy_data selftest
import sys, time
REPO = "/Workspace/Users/<your-databricks-user>/datathon"
sys.path.insert(0, REPO)
sys.path.insert(0, REPO + "/baseline-bots")
from evaluator import warmup
from bot import strategy
from train import selfplay
warmup()

mean, lo, hi = selfplay.match(strategy.load_params(), "add3", 100, "check")
print(f"match vs add3 (100 deals): {mean:+.3f} bb/deal  CI [{lo:+.3f}, {hi:+.3f}]")

from train import policy_data
t0 = time.perf_counter()
n = policy_data.selftest()
print(f"policy_data.selftest(): {n} rows, {time.perf_counter() - t0:.1f} s")

In [0]:
# G1+G2: one work unit = generate_labeled(seed, opp, n_hands=200, k=8).
# Opponents drawn by weight from policy_data.pool() with a fixed driver RNG (seed "db-draw-1").
import random
import pandas as pd

pool = policy_data.pool()
names, weights = zip(*[(n, pool[n][1]) for n in pool])
TOTAL = 2760  # ~200k rows at ~74 rows/unit
rng = random.Random("db-draw-1")
opps = rng.choices(list(names), weights=list(weights), k=TOTAL)

def run_units(batches):
    import sys
    import pandas as pd
    REPO = "/Workspace/Users/<your-databricks-user>/datathon"
    sys.path.insert(0, REPO)
    sys.path.insert(0, REPO + "/baseline-bots")
    from train import policy_data
    from evaluator import warmup
    warmup()
    for pdf in batches:
        rows = []
        for r in pdf.itertuples(index=False):
            for row in policy_data.generate_labeled(r.seed, r.opp, n_hands=200, k=8):
                rows.append((row["seed"], row["opponent"], int(row["hand_index"]), row["street"],
                             int(row["facing_raise"]), int(row["pot"]),
                             [float(x) for x in row["features"]],
                             [float(x) for x in row["ev"]],
                             [float(x) for x in row["q"]],
                             [float(x) for x in row["q_std"]]))
        yield pd.DataFrame(rows, columns=["seed", "opponent", "hand_index", "street", "facing_raise",
                                         "pot", "features", "ev", "q", "q_std"])

def write_batch(lo, hi, mode):
    spec = pd.DataFrame({"seed": [f"d{i+1}" for i in range(lo, hi)], "opp": opps[lo:hi]})
    (spark.createDataFrame(spec).repartition(min(200, hi - lo))
          .mapInPandas(run_units,
                       "seed string, opponent string, hand_index int, street string, facing_raise int, pot int, "
                       "features array<double>, ev array<double>, q array<double>, q_std array<double>")
          .write.mode(mode).format("delta").saveAsTable("workspace.default.policy_rows"))
    print(lo + 1, "..", hi, "->", spark.read.table("workspace.default.policy_rows").count(), "rows")

In [0]:
# Cost pilot: 20 units, 4 partitions (NOT persisted — seeds d1..d20 are re-run and written below)
t0 = time.perf_counter()
pilot_rows = (spark.createDataFrame(pd.DataFrame({"seed": [f"d{i+1}" for i in range(20)], "opp": opps[:20]}))
       .repartition(4)
       .mapInPandas(run_units,
                    "seed string, opponent string, hand_index int, street string, facing_raise int, pot int, "
                    "features array<double>, ev array<double>, q array<double>, q_std array<double>")).count()
print(f"pilot: 20 units -> {pilot_rows} rows in {time.perf_counter()-t0:.0f} s (72.6 rows/unit, 2.2 s/unit)")

In [0]:
# Full run: 2,760 units in 3 append batches (keeps each Spark action short).
# Actual wall times: 756 s + 738 s + 574 s (~35 min total, ~6.9 core-hours at k=8).
write_batch(0, 1000, "overwrite")
write_batch(1000, 2000, "append")
write_batch(2000, 2760, "append")

In [0]:
%sql
-- Rows by street and facing_raise (TURN/RIVER facing a raise are oversampled 3x by policy_data.OVERSAMPLE)
SELECT street, COUNT(*) n, SUM(facing_raise) facing_raise_rows, ROUND(AVG(pot), 1) avg_pot
FROM workspace.default.policy_rows GROUP BY street
ORDER BY CASE street WHEN 'PREFLOP' THEN 0 WHEN 'FLOP' THEN 1 WHEN 'TURN' THEN 2 ELSE 3 END;

In [0]:
%sql
-- Rows by opponent (strong opponents, incl. pool/v* and var/*, count double in the draw weights)
SELECT opponent, COUNT(*) n, ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) pct
FROM workspace.default.policy_rows GROUP BY opponent ORDER BY n DESC;

In [0]:
# G3: broken rows, advantage stats per action/street, FOLD-best facing a raise, noise floor.
# Round-1 findings: 0 broken rows; advantage SD 19.6-25.4 chips by street; FOLD best on 45-58% of
# facing-raise spots (rising by street); noise floor q_std/sqrt(8) 0.7-3.7 chips; corr(EV,Q)=0.294.
import math
import numpy as np

pdf = spark.read.table("workspace.default.policy_rows").toPandas()
sys.path.insert(0, REPO)
from game import Action
ACTIONS = [a.name for a in Action]  # FOLD, CHECK_CALL, RAISE_HALF, RAISE_POT
n = len(pdf)
Q = np.array([[float(x) for x in r] for r in pdf.q])
EV = np.array([[float(x) for x in r] for r in pdf.ev])
QS = np.array([[float(x) for x in r] for r in pdf.q_std])
legal = ~np.isnan(Q)
broken = int((~legal[:, 1]).sum() + (legal.sum(axis=1) == 0).sum()
             + pdf.pot.isna().sum() + (pdf.features.apply(len) != 30).sum())
print(f"rows {n}, broken {broken} ({broken / n * 100:.2f}%) -> {'STOP' if broken > 0.01 * n else 'OK'}")

adv = Q - np.where(legal, np.nanmean(np.where(legal, Q, np.nan), axis=1, keepdims=True), np.nan)
for i, name in enumerate(ACTIONS):
    v = adv[:, i][legal[:, i]]
    print(f"adv {name:10s} n={len(v):6d}  mean {v.mean():+7.2f}  sd {v.std():6.2f}")
for st in ("PREFLOP", "FLOP", "TURN", "RIVER"):
    m = (pdf.street == st).values
    v = adv[m].ravel(); v = v[~np.isnan(v)]
    nf = (QS[m] / math.sqrt(8)).ravel(); nf = nf[~np.isnan(nf)]
    mr = ((pdf.street == st) & (pdf.facing_raise == 1)).values
    best = np.nanargmax(np.where(legal & mr[:, None], Q, -np.inf), axis=1)[mr]
    print(f"{st:8s} adv mean {v.mean():+5.1f} sd {v.std():5.1f} | noise {nf.mean():4.1f} | FOLD best when raised {np.mean(best == 0) * 100:4.1f}%")

In [0]:
# G4: port of train/policy_train.py — its ridge/mlp/forward/rows_of functions are used UNCHANGED.
# Split by seed (zlib.crc32(seed)%5==0 -> 20% held out). Target = masked advantage (q - mean over legal).
# Round-1: train 163,799 / test 41,355; ridge ~0 s, each MLP ~67 s.
import zlib, shutil
for m in [m for m in list(sys.modules) if m.split(".")[0] in ("bot", "train", "selfplay", "tune", "opponents",
                                                            "strategy", "policy_data", "policy_train",
                                                            "evaluator", "game", "opponent")]:
    del sys.modules[m]
sys.path.insert(0, REPO + "/train")
import policy_train as pt
warmup()

F = np.array([list(map(float, r)) for r in pdf.features])
POT = pdf.pot.to_numpy(float)
SEED = np.array([str(s) for s in pdf.seed])
mask = ~np.isnan(Q)
adv = np.where(mask, Q - np.nanmean(np.where(mask, Q, np.nan), axis=1, keepdims=True), 0.0)
test = np.array([zlib.crc32(s.encode()) % 5 == 0 for s in SEED])
mu, sd = F[~test].mean(0), F[~test].std(0); sd[sd < 1e-6] = 1.0
Z = (F - mu) / sd
tr = ~test
temp = pt.strategy.load_params()["temperature"]
print(f"train rows {tr.sum()}, test rows {test.sum()} (split by seed)")

members = {"lin": pt.ridge(Z[tr], adv[tr], mask[tr])}
for i in range(5):
    members[f"mlp{i}"] = pt.mlp(Z[tr], adv[tr], mask[tr], seed=i + 1)  # bootstrap bags, seeds 1..5
preds = {k: pt.forward(v, Z[test]) for k, v in members.items()}
ens = np.mean(list(preds.values()), axis=0)
d = {"q": Q, "ev": EV, "pot": POT, "seed": SEED}
for name, pred in {**preds, "ensemble": ens}.items():
    pt.report(name, pred, adv[test], mask[test], d, test, temp)

In [0]:
# Log every member + the plain average to MLflow experiment poker_policy_ensemble (round 1).
import mlflow
mlflow.autolog(disable=True)
mlflow.set_experiment("poker_policy_ensemble")
tv = int(spark.sql("DESCRIBE HISTORY workspace.default.policy_rows").first()["version"])
for name, pred in {**preds, "ensemble": ens}.items():
    mae = np.abs(pred - adv[test])[mask[test]].mean()
    big = (np.where(mask[test], pred, -np.inf).argmax(1)
           == np.where(mask[test], adv[test], -np.inf).argmax(1)).mean()
    rows = pt.rows_of(d, test)
    with mlflow.start_run(run_name=f"round1_{name}"):
        mlflow.log_param("member", name)
        mlflow.log_param("data_table", "workspace.default.policy_rows")
        mlflow.log_param("table_version", tv)
        mlflow.log_param("train_rows", int(tr.sum()))
        mlflow.log_param("test_rows", int(test.sum()))
        mlflow.log_metric("adv_mae", float(mae))
        mlflow.log_metric("best_action_match", float(big))
        for w in (0.0, 0.25, 0.5, 1.0, 2.0):
            mlflow.log_metric(f"offline_value_w{w}", float(pt.offline_value(rows, pred, w, temp)))

In [0]:
# Whole-ensemble predict latency, single row, single CPU: round-1 p50 0.063 ms / p95 0.065 ms.
def predict_one(f):
    x = (np.asarray(f, dtype=np.float64) - mu) / sd
    o = np.zeros(4)
    for layers in members.values():
        a = x
        for i, (W_, b_) in enumerate(layers):
            a = a @ W_ + b_
            if i < len(layers) - 1:
                a = np.maximum(a, 0.0)
        o += a / len(members)
    return o
ts = []
for f in F[test][:500]:
    t0 = time.perf_counter(); predict_one(f); ts.append(time.perf_counter() - t0)
print(f"ensemble latency: p50={np.median(ts)*1000:.3f} ms  p95={np.percentile(ts, 95)*1000:.3f} ms")

In [0]:
# G5: export bot/policy_weights.npz (ridge + mlp0..4 + mu/sd, format bot/policy_model.py reads — file unchanged)
# and bot/policy_parity.npz (1,000 held-out test rows). Fresh-process check runs in a fresh executor
# python worker via a single-partition mapInPandas.
# Round-1: parity max|diff| = 0 over 1,000 rows (tolerance 1e-5); predict 0.038 ms p50/p95.
out = {"mu": mu, "sd": sd, "members": np.array(list(members)), "alpha": np.ones(len(members))}
for k, layers in members.items():
    for i, (W_, b_) in enumerate(layers):
        out[f"{k}/W{i}"], out[f"{k}/b{i}"] = W_, b_
np.savez("/tmp/policy_weights.npz", **out)

sel = np.where(test)[0][:1000]
Zp = (F[sel] - mu) / sd
exp = np.mean([pt.forward(v, Zp) for v in members.values()], axis=0)
np.savez("/tmp/policy_parity.npz", features=F[sel], expected=exp)
shutil.copy2("/tmp/policy_weights.npz", REPO + "/bot/policy_weights.npz")
shutil.copy2("/tmp/policy_parity.npz", REPO + "/bot/policy_parity.npz")

par = pd.DataFrame({"features": [F[i] for i in sel], "expected": [exp[i] for i in range(len(sel))]})
def check(batches):
    import sys, time
    import numpy as np
    sys.path.insert(0, "/Workspace/Users/<your-databricks-user>/datathon")
    from bot import policy_model  # fresh import: loads the exported weights
    for p_ in batches:
        maxdiff, t_ = 0.0, []
        for f, e in zip(p_.features, p_.expected):
            t0 = time.perf_counter(); o = policy_model.predict(f); t_.append(time.perf_counter() - t0)
            maxdiff = max(maxdiff, float(np.max(np.abs(np.asarray(o) - np.asarray(e)))))
        yield pd.DataFrame([{"max_diff": maxdiff, "p50_ms": float(np.median(t_)) * 1000,
                             "p95_ms": float(np.percentile(t_, 95)) * 1000, "n": len(t_)}])
res = (spark.createDataFrame(par).repartition(1)
       .mapInPandas(check, "max_diff double, p50_ms double, p95_ms double, n int").toPandas())
print(res.to_string(index=False))

## G6: handover

`bot/policy_weights.npz`, `bot/policy_parity.npz` and this notebook are committed to branch
**`db-policy-model`**. No other files changed; `use_policy_model` stays off in `bot/params.json`
until the local gate (`train/compare.py`) accepts it. Local step 7: pull the branch, parity test,
latency test, accept gate, 50 matches vs `spoc`, Docker check. Round 2 regenerates data with the
accepted policy under fresh seed prefixes.

MLflow experiment `poker_policy_ensemble` (round 1, data table v8): lin `40eb968c0e89427d90a46dee7e1b1b1a`,
mlp0 `b1d8afcf5ecd4781b0c0d4b6fb0b6852`, mlp1 `4a4f606b72944dc7912e3e3a38956f40`,
mlp2 `2b40aa6d247445018d68c015b5d017fa`, mlp3 `aaf9143844d3415cad04caf51362498c`,
mlp4 `5df6f1dd4c2e4bb9980167f5b9d7882b`, ensemble `24551c7a561c4ccfa06b88baebda7905`.

Offline value on held-out seeds (chips/decision, `policy_data.offline_value`): rules-only w=0 +3.286,
w=0.25 +3.717, w=0.5 +3.936, **w=1 +4.134**, w=2 +4.290 — the blend dominates the rules at every tested weight.